In [1]:
import xarray as xr 
from anemoi.datasets import open_dataset
import numpy as np
import yaml
import os 
import cfgrib
from pathlib import Path
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import pathlib
import csv
import re
import matplotlib.patches as mpatches
import pandas as pd

In [8]:
xr.open_dataset("/mnt/weatherloss/WindPower/data/cerra_boz/raw_grib/cerra_pressure_2022_10.grib",engine="cfgrib").longitude.values

array([[342.514057  , 342.55818325, 342.60232794, ...,  33.39767134,
         33.44181602,  33.48594228],
       [342.49840486, 342.54255389, 342.58672138, ...,  33.41327789,
         33.45744539,  33.50159442],
       [342.48273423, 342.52690606, 342.57109638, ...,  33.4289029 ,
         33.47309322,  33.51726504],
       ...,
       [302.06206865, 302.13088553, 302.19985614, ...,  73.80014273,
         73.86911335,  73.93793022],
       [301.97856972, 302.04735663, 302.1162975 , ...,  73.88370137,
         73.95264224,  74.02142915],
       [301.89491724, 301.96367379, 302.03258451, ...,  73.96741436,
         74.03632508,  74.10508164]])

## Lower resolution on CERRA

In [2]:
cerra_outer=xr.open_dataset("cerra_outer.zarr")
def outline_lonlat(ds):
    lat = ds["latitude"].values
    lon = ds["longitude"].values

    edge_lon = np.concatenate([
        lon[0, :],
        lon[:, -1],
        lon[-1, ::-1],
        lon[::-1, 0],
    ])
    edge_lat = np.concatenate([
        lat[0, :],
        lat[:, -1],
        lat[-1, ::-1],
        lat[::-1, 0],
    ])
    return edge_lon, edge_lat


FileNotFoundError: No such file or directory: '/mnt/weatherloss/WindPower/data/NorthSea/Cerra/cerra_outer.zarr'

In [3]:
ds = xr.open_dataset("cerra_outer.zarr", engine="zarr")

lon_min, lon_max = -4, 10
lat_min, lat_max = 50.75, 57.0

lon2d = ds["longitude"].values
lat2d = ds["latitude"].values

mask = (
    (lon2d >= lon_min) & (lon2d <= lon_max) &
    (lat2d >= lat_min) & (lat2d <= lat_max)
)


yy, xx = np.where(mask)
y0, y1 = int(yy.min()), int(yy.max())
x0, x1 = int(xx.min()), int(xx.max())

# Inner high-res dataset (still CERRA grid, just smaller)
#cerra_HR = ds.isel(y=slice(y0, y1+1), x=slice(x0, x1+1))
#cerra_HR.to_zarr("cerra_CI_HR.zarr",mode="w")


In [4]:
# factor = 5  # ~27 km

# # coarsen all variables that live on (y,x)
# vars_xy = [v for v in ds.data_vars if {'y','x'}.issubset(ds[v].dims)]
# ds_xy = ds[vars_xy].coarsen(y=factor, x=factor, boundary="trim").mean()

# # coarsen coords too
# lat_c = ds["latitude"].coarsen(y=factor, x=factor, boundary="trim").mean()
# lon_c = ds["longitude"].coarsen(y=factor, x=factor, boundary="trim").mean()

# cerra_LR = ds_xy.assign_coords(latitude=lat_c, longitude=lon_c)
cerra_LR=xr.open_dataset("cerra_LR.zarr")
#cerra_outer_coarse.to_zarr("cerra_outer_coarse.zarr", mode="w"

In [5]:

meta_path = Path("/mnt/weatherloss/WindPower/data/NorthSea/Power/windfarm_metadata.csv")
farms = []
with meta_path.open("r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        farms.append((float(row["lat"]), float(row["lon"])))
farm_lats, farm_lons = zip(*farms)
farm_lons = np.array(farm_lons)

varname = "ws100"  # change if needed
da_lr = cerra_LR[varname]
da_hr = cerra_HR[varname]

if "time" in da_lr.dims:
    t_idx = np.random.randint(da_lr.sizes["time"])
    da_lr = da_lr.isel(time=t_idx)
    da_hr = da_hr.isel(time=t_idx)
    timestamp = str(cerra_LR["time"].isel(time=t_idx).values)

lon_lr = cerra_LR["longitude"].values
if np.nanmax(lon_lr) > 180:
    farm_lons_plot = np.where(farm_lons < 0, farm_lons + 360, farm_lons)
else:
    farm_lons_plot = farm_lons

# ---- plot ----
fig = plt.figure(figsize=(10, 9))
ax = plt.axes(projection=ccrs.PlateCarree())

# extent to LR domain
lon_lr = cerra_LR["longitude"].values
lat_lr = cerra_LR["latitude"].values

lon_hr = cerra_HR["longitude"].values
lat_hr = cerra_HR["latitude"].values

lon_min = np.nanmin([lon_lr.min(), lon_hr.min()])
lon_max = np.nanmax([lon_lr.max(), lon_hr.max()])
lat_min = np.nanmin([lat_lr.min(), lat_hr.min()])
lat_max = np.nanmax([lat_lr.max(), lat_hr.max()])

# LR background
da_lr.plot(
    ax=ax, x="longitude", y="latitude",
    transform=ccrs.PlateCarree(),
    cmap="coolwarm",
    alpha=0.6,
    zorder=1,
    add_colorbar=True,
    cbar_kwargs=dict(
        shrink=0.5,      # <--- main fix
        aspect=30,       # thinner bar
        pad=0.02,        # distance from map
        label="Wind speed (m/s)"
    ),
)

# HR overlay
da_hr.plot(
    ax=ax, x="longitude", y="latitude",
    transform=ccrs.PlateCarree(),
    cmap="coolwarm",
    zorder=3,
    add_colorbar=False,
)

# outlines
lr_lon, lr_lat = outline_lonlat(cerra_LR)
hr_lon, hr_lat = outline_lonlat(cerra_HR)


ax.plot(lr_lon, lr_lat, color="black", linewidth=2,
        transform=ccrs.PlateCarree(), label="Inner extent", zorder=4)
ax.plot(hr_lon, hr_lat, color="lime", linewidth=2,
        transform=ccrs.PlateCarree(), label="Outer extent", zorder=5)

# # wind farms
# ax.scatter(
#     farm_lons_plot, farm_lats,
#     color="red", s=12,
#     transform=ccrs.PlateCarree(),
#     zorder=6,
#     label="Wind farms"
# )

# map features
ax.coastlines(zorder=7)
ax.add_feature(cfeature.BORDERS, linewidth=0.7, zorder=7)

ax.legend(loc="lower left")
ax.set_title(f"100-meter wind speed (random timestamp)",fontsize=16)
#ax.set_title(f"CERRA LR vs HR ({varname}) + wind farms @ {timestamp}")

plt.tight_layout()
#plt.savefig("cerra_CI_map.png")
plt.show()


NameError: name 'cerra_HR' is not defined

In [ ]:
cerra_HR.to_zarr("cerra_CI_HR.zarr",mode="w")

In [ ]:
cerra_LR.to_zarr("cerra_CI_LR.zarr",mode="w")

/tmp/ipykernel_21830/2284907985.py:1: SerializationWarning: saving variable None with floating point data as an integer dtype without any _FillValue to use for NaNs
  cerra_LR.to_zarr("cerra_LR.zarr",mode="w")


In [14]:
ds=xr.open_dataset("cerra_HR.zarr")
ds["capacity"].sel(x=103,y=57).values

array(630., dtype=float32)

In [5]:
cerra_CI_HR=xr.open_dataset("cerra_CI_HR.zarr")

In [28]:
LR=xr.open_dataset("cerra_LR.zarr")
LR['latitude'].values

array([[48.28935122, 48.33345153, 48.37636188, ..., 49.07363689,
        49.0635299 , 49.05217765],
       [48.53246662, 48.57678868, 48.61991503, ..., 49.3207338 ,
        49.31057498, 49.29916453],
       [48.77556731, 48.82011248, 48.86345618, ..., 49.56784139,
        49.5576304 , 49.54616137],
       ...,
       [57.95148163, 58.00571076, 58.05849291, ..., 58.91848814,
        58.90599076, 58.89195476],
       [58.1898944 , 58.24441618, 58.29748376, ..., 59.16221075,
        59.14964349, 59.13552903],
       [58.42807226, 58.48288903, 58.53624431, ..., 59.40574263,
        59.39310487, 59.37891128]])

In [7]:
ds=xr.open_dataset("/mnt/weatherloss/WindPower/data/NorthSea/Cerra/cerra_CI_HR.zarr")
ds

<xarray.Dataset> Size: 102GB
Dimensions:               (y: 137, x: 178, time: 16312, level: 9)
Coordinates:
  * y                     (y) int64 1kB 36 37 38 39 40 ... 168 169 170 171 172
  * x                     (x) int64 1kB 32 33 34 35 36 ... 205 206 207 208 209
  * time                  (time) datetime64[ns] 130kB 2020-01-01 ... 2025-07-...
  * level                 (level) float64 72B 1e+03 950.0 900.0 ... 600.0 500.0
    latitude              (y, x) float64 195kB ...
    longitude             (y, x) float64 195kB ...
Data variables: (12/31)
    capacity              (y, x) float32 98kB ...
    doy_cos               (time) float32 65kB ...
    doy_sin               (time) float32 65kB ...
    lsm                   (y, x) float32 98kB ...
    mask                  (y, x) float32 98kB ...
    mcc                   (time, y, x) float32 2GB ...
    ...                    ...
    ws10                  (time, y, x) float32 2GB ...
    ws100                 (time, y, x) float32 2GB ...
    ws150                 (time, y, x) float32 2GB ...
    ws200                 (time, y, x) float32 2GB ...
    ws50                  (time, y, x) float32 2GB ...
    z                     (time, level, y, x) float32 14GB ...

In [35]:
ds['power'].isel(time=15000).values

array([[ 265.97034 ,   35.200092,  136.61125 , ...,         nan,
                nan,         nan],
       [ 226.52783 ,  151.18404 ,  187.45454 , ...,         nan,
                nan,         nan],
       [ 222.06464 , 1443.7946  ,  161.11209 , ...,         nan,
                nan,         nan],
       ...,
       [        nan,         nan,         nan, ...,   82.184425,
          96.676834,   20.619541],
       [        nan,         nan,         nan, ...,   35.668453,
          18.4127  ,   30.286285],
       [        nan,         nan,         nan, ...,   47.53009 ,
          21.923906,   30.897903]], dtype=float32)

In [3]:
ds=xr.open_dataset("/mnt/weatherloss/WindPower/data/NorthSea/Cerra/cerra_CI_HR.zarr")
ds

<xarray.Dataset> Size: 102GB
Dimensions:               (y: 137, x: 178, time: 16312, level: 9)
Coordinates:
  * y                     (y) int64 1kB 36 37 38 39 40 ... 168 169 170 171 172
  * x                     (x) int64 1kB 32 33 34 35 36 ... 205 206 207 208 209
  * time                  (time) datetime64[ns] 130kB 2020-01-01 ... 2025-07-...
  * level                 (level) float64 72B 1e+03 950.0 900.0 ... 600.0 500.0
    latitude              (y, x) float64 195kB ...
    longitude             (y, x) float64 195kB ...
Data variables: (12/31)
    capacity              (y, x) float32 98kB ...
    doy_cos               (time) float32 65kB ...
    doy_sin               (time) float32 65kB ...
    lsm                   (y, x) float32 98kB ...
    mask                  (y, x) float32 98kB ...
    mcc                   (time, y, x) float32 2GB ...
    ...                    ...
    ws10                  (time, y, x) float32 2GB ...
    ws100                 (time, y, x) float32 2GB ...
    ws150                 (time, y, x) float32 2GB ...
    ws200                 (time, y, x) float32 2GB ...
    ws50                  (time, y, x) float32 2GB ...
    z                     (time, level, y, x) float32 14GB ...

In [26]:
ds=xr.open_dataset("/mnt/weatherloss/WindPower/inference/CI/Transformer/forecast_20240801090000.nc")

In [25]:
ds.latitude.values

array([50.197445, 50.205185, 50.21288 , ..., 59.405743, 59.393105,
       59.37891 ], dtype=float32)

In [35]:
ds=xr.open_dataset("/mnt/weatherloss/WindPower/data/NorthSea/Cerra/cerra_CI_HR.zarr")
ds

<xarray.Dataset> Size: 102GB
Dimensions:               (y: 137, x: 178, time: 16312, level: 9)
Coordinates:
  * y                     (y) int64 1kB 36 37 38 39 40 ... 168 169 170 171 172
  * x                     (x) int64 1kB 32 33 34 35 36 ... 205 206 207 208 209
  * time                  (time) datetime64[ns] 130kB 2020-01-01 ... 2025-07-...
  * level                 (level) float64 72B 1e+03 950.0 900.0 ... 600.0 500.0
    latitude              (y, x) float64 195kB ...
    longitude             (y, x) float64 195kB ...
Data variables: (12/31)
    capacity              (y, x) float32 98kB ...
    doy_cos               (time) float32 65kB ...
    doy_sin               (time) float32 65kB ...
    lsm                   (y, x) float32 98kB ...
    mask                  (y, x) float32 98kB ...
    mcc                   (time, y, x) float32 2GB ...
    ...                    ...
    ws10                  (time, y, x) float32 2GB ...
    ws100                 (time, y, x) float32 2GB ...
    ws150                 (time, y, x) float32 2GB ...
    ws200                 (time, y, x) float32 2GB ...
    ws50                  (time, y, x) float32 2GB ...
    z                     (time, level, y, x) float32 14GB ...

In [36]:
import xarray as xr
import numpy as np
import pandas as pd

COORD_DECIMALS = 5  # try 4 then 5

def ll_index(lat, lon):
    lat = np.round(np.asarray(lat, float), COORD_DECIMALS)
    lon = np.round(np.asarray(lon, float), COORD_DECIMALS)
    return pd.MultiIndex.from_arrays([lat, lon], names=["lat","lon"])

path = "/mnt/weatherloss/WindPower/inference/CI/Transformer/forecast_20240801090000.nc" # adjust to an existing file
ds = xr.open_dataset(path)

ll = ll_index(ds["latitude"].values, ds["longitude"].values)
print("values:", len(ll))
print("unique:", ll.nunique())
print("duplicates:", len(ll) - ll.nunique())

ds.close()


values: 25427
unique: 25427
duplicates: 0


In [38]:
137*178

24386